In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
my_schema = """order_id STRING,
        timestamp STRING,
        customer STRUCT<
          customer_id: INT,
          name: STRING,
          email: STRING,
          address: STRUCT<
            city: STRING,
            postal_code: STRING,
            country: STRING
          >
        >,
        items ARRAY<STRUCT<
          item_id: STRING,
          product_name: STRING,
          quantity: INT,
          price: DOUBLE
        >>,
        payment STRUCT<
          method: STRING,
          transaction_id: STRING
        >,
        metadata ARRAY<STRUCT<
          key: STRING,
          value: STRING
        >>"""

In [0]:
day_1_json = "/Volumes/dbcertification/streaming/streamingdata/input/day1.json"

In [0]:
df = (spark.read
      .format("json")
      .option("multiline",True)
      .option("inferSchema",True)
      .load(day_1_json)
      )

In [0]:
display(df.limit(5))

In [0]:
from pyspark.sql.functions import inline_outer

df_exploded = df.select("customer.*","*","payment.*").select("address.*","*",inline_outer("items"),inline_outer("metadata")).drop("customer","address","items","metadata","payment")

In [0]:
display(df_exploded.limit(5))

#### STREAMING READ

In [0]:
json_source = "/Volumes/dbcertification/streaming/streamingdata/input/"

In [0]:
df_stream = (spark.readStream
      .format("json")
      .option("multiline",True)
      .schema(my_schema)
      .load(json_source)
      )

In [0]:
from pyspark.sql.functions import inline_outer

df_stream_exploded = df_stream.select("customer.*","*","payment.*").select("address.*","*",inline_outer("items"),inline_outer("metadata")).drop("customer","address","items","metadata","payment")

In [0]:
(
df_stream_exploded.writeStream.format("delta")
    .outputMode("append") 
    .trigger(once=True)
    .option("path","/Volumes/dbcertification/streaming/streamingdata/jsonsink/Data")
    .option("checkpointLocation","/Volumes/dbcertification/streaming/streamingdata/jsonsink/checkpoint")
    .start()
 )
 # once = True means : whatever data you have you just need to process that

In [0]:
%sql
select * from delta.`/Volumes/dbcertification/streaming/streamingdata/jsonsink/Data`

- On first load only day_1.json was present so it got processed
- On Second load one more json was present in input folder and only day_2.json was processed. This is bcos of idempotency.
- **Idempotency** : That means data which is already processed will not be processed one more time, that is biggest advantage of streaming. 
- Spark Streaming handles it internally using checkpoint.
- If we re-run the query now, nothing will be processed, although we have 2 files but none will be processed again.
- sources folder : keeps track of full path of files already picked up, so that it doesnt get processed again.
- offsets folder : it simply assigns index to data which is read.
- commits folder : its like checkmark, this file successfully processed tick. Due to any reason if file was not successfully written to destination. So there will not be any commit version of it. So in next batch it will re process the data which was failed.
 

#### Triggers in Streaming:
- **Default** :  Depends on processing time previous batch takes. Then only next batch is picked up. 
- **Processing Time** : Will check source suppose every 10 second or whatever time thats been given. Will check every 'x' second or minute as mentioned - **.trigger(processingTime="10 seconds")**. It will be continously checking without stopping.
- **Once** : We need to process all the data present in source and then stop the Trigger. Will do it in one micro batch all the data processed at once.
- **Available Now** : It will also read all the data but will divide processing into small-small micro batches. Will keep on doing until all data is processed, so there will be less load on processing. It is better version of Once Trigger.
- **Continuous** : Recently added, not compatible with all the data sources and sinks. It is only compatible with Memory, Console, Kafka and it only supports "Append Only" mode. It is built on concept of processing data Row-By-Row instead of processing data in batch. **.trigger(continuous="1 second")** Since it is processing data continously so why this 1 second. This 1 second is for Checkpoint location. It will update the checkpoint location in every 1 second.
     - We know once a microbatch is complete it will update the checkpoint location but here there is no micro-batch.  
     - It is really faster than other triggers.


#### ARCHIVING
- **ARCHIVING the Source Files :**
- One Archive directory is required to put processed file.
- Moment we receive new file and is going to be processed, then old processed file is moved to archive folder.
- It promotes readability
- Processing time will be much quicker. Bcos it will read only latest files.
- If i re-upload processed file. what will happen? Nothing will happen. No processing will happen. And since no processing happens, existing and new both processed files are not moved to archive folder.
- Next if we have new file which is not yet processed then only first processed file is moved to folder and duplicate file will be present in source folder itself because it is garbage now.

#### Implementing Archiving

In [0]:
from pyspark.sql.functions import inline_outer

json_source_archive = "/Volumes/dbcertification/streaming/streamingdata/inputsourcearchive/"
json_archive = "/Volumes/dbcertification/streaming/streamingdata/inputarchive/"

df_stream_arch = (spark.readStream
      .format("json")
      .option("multiline",True)
      .schema(my_schema)
      .option("cleanSource","archive")
      .option("sourceArchiveDir",json_archive)
      .load(json_source_archive)
      )


df_stream_arch_exploded = df_stream_arch.select("customer.*","*","payment.*").select("address.*","*",inline_outer("items"),inline_outer("metadata")).drop("customer","address","items","metadata","payment")

In [0]:

(df_stream_arch_exploded.writeStream.format("delta")
    .outputMode("append") 
    .trigger(once=True)
    .option("path","/Volumes/dbcertification/streaming/streamingdata/jsonsinkarch/Data")
    .option("checkpointLocation","/Volumes/dbcertification/streaming/streamingdata/jsonsinkarch/checkpoint")
    .start()
)
 # once = True means : whatever data you have you just need to process that

**Note** : 
 - **File Immutability Assumption**: Under the default configuration, Auto Loader and Structured Streaming operate under the strict assumption that files in the source directory are immutable (they never change once written).


 - **Path-Based Tracking**: The system checks its checkpoint state and sees that the file path .../Test1.json has already been recorded as processed. Because it assumes the file cannot change, it does not bother scanning the inside of the file to check if the content is different. It simply ignores it and skips to the next file.

 - If you want the system to actually check for changes and reprocess the file when its content or modification time changes, you must explicitly enable the overwrite option:
 - **.option("cloudFiles.allowOverwrites", "true")**

 - Without explicitly setting that option to true, the new Test1.json will be completely ignored by the stream forever, regardless of what data is inside it.